# mart.team_failures

A Saved table the batch team_failures made, with project.convert(...), from the statements in `conversion/statement.hql`, as its script wrote them. Its logic cell builds the same query from the Toolbox's own functions, with create(), preview() and write(). Tick each box of `conversion/checklist.md`, then run the cells top to bottom. The cells comparing it with the original stay until you delete them.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # the Project: two up

from sqlglot_composer import (compare_hive, compare_with_original, export_lineage, project,
                              run_query, show_hive)
from warehouse import query

In [ ]:
# logic
from sqlglot_composer import (
    AS,
    Clause,
    FROM,
    GROUP_BY,
    JOIN,
    SELECT,
    Statement,
    WHERE,
    all_of,
    between,
    count_rows,
    derived,
    equals,
    if_else,
    is_in,
    mapped,
    statement,
    sum_of,
)
from references import ops
from sqlglot_composer import create_table  # for create()
from sqlglot_composer import INSERT_OVERWRITE, Statement, statement  # for write()
from references.mart__team_failures import team_failures  # the Saved table's own Table reference

EVENT_TYPE_VALUES = ["start", "retry", "fail", "finish"]
TEAM_NAME_BY_TEAM = {
    "data": "Data platform",
    "web": "Web",
}

e = AS(ops.job_events, "e")
ev = derived("ev", statement(
    SELECT(
        e.job_id,
        e.dt,
        e.event_type,
    ),
    FROM(e),
    WHERE(all_of(
        between(e.dt, "2026-09-20", "2026-09-24"),
        is_in(e.event_type, EVENT_TYPE_VALUES),
    )),
))
o = AS(ops.job_owners, "o")
per_team = derived("per_team", statement(
    SELECT(
        o.team,
        ev.dt,
        AS(count_rows(), "events"),
        AS(sum_of(if_else(equals(ev.event_type, "fail"), 1, 0)), "fails"),
    ),
    FROM(ev),
    JOIN(o, ON=all_of(
        equals(o.job_id, ev.job_id),
        equals(o.dt, ev.dt),
    ), reads_all_partitions=True),
    GROUP_BY(o.team, ev.dt),
))
p = AS(per_team, "p")


def rows() -> list[Clause]:
    """The clauses of the query, which converted() and a write share."""
    return [
        SELECT(
            p.team,
            AS(mapped(p.team, TEAM_NAME_BY_TEAM, otherwise="Other"), "team_name"),
            p.events,
            p.fails,
            p.dt,
        ),
        FROM(p),
    ]


def converted() -> Statement:
    """The query, as the Hive this Conversion was made from writes it."""
    return statement(*rows())


def create() -> Statement:
    """Create mart.team_failures from its Table reference, if it isn't there yet."""
    return create_table(team_failures, may_exist=True)


def preview() -> Statement:
    """The rows write() would save, as a SELECT to run and check first."""
    return converted()


def write() -> Statement:
    """Save the rows into mart.team_failures, as the original's INSERT did."""
    return statement(INSERT_OVERWRITE(team_failures), *rows())


In [ ]:
# Try it: its rows, then, once its Table reference has its location=, the Hive of
# each step
run_query(preview(), query)
show_hive([create(), write()])

In [ ]:
# Lineage: a new pair in lineage/, named by the time and this table
export_lineage(write())

In [ ]:
# Its Hive against the original's, as conversion/in_dev.hql holds it
ORIGINAL = pathlib.Path("conversion/in_dev.hql").read_text(encoding="utf-8")
show_hive(preview())
print(compare_hive(preview(), ORIGINAL))

In [ ]:
# The convert Workflow's step 6, project.compare_batch, compares this table once written.
# This compares its query with the original's first:
# Its rows against the original's, through your query, on one Partition:
# print(compare_with_original(preview(), ORIGINAL, query=query, partition={"dt": "<a value>"}))

In [ ]:
# Its rows in the Reference sheet
project.refresh_reference_sheet("mart.team_failures")